# EXXA GSoC 2026 — General Test
**Author:** Viren Pandey

Unsupervised clustering of synthetic ALMA 1250µm protoplanetary disk observations.

In [ ]:
!pip install astropy gdown umap-learn -q

import os, glob
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from astropy.io import fits
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import umap, warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
import gdown
DATA_DIR = '/content/exxa_data'
os.makedirs(DATA_DIR, exist_ok=True)

FILE_IDS = [
    ('1Ozq_DRVWXEyY7Q_VFet5cJPn33YLKNpK','planet0_00226_1250.fits'),
    ('1LxF4W7Ecyfa4nYOPbPTAYedL8JEuTN2q','planet1_00490_1250.fits'),
    ('1pH4Pvap9yqO6v6riV0cSE42MQotV2m1i','planet2_00154_1250.fits'),
    ('1CLktdDuk2OJJ0bvxZy9kN7OTq4sdXabB','planet2_00308_1250.fits'),
    ('1PumICVIGBtk483UvZhHTLDK8M4X2MQVU','planet2_00548_1250.fits'),
    ('1TDycE_Xjjw6Q8Tlb5AoPy5GAtGtCKu8u','planet2_00708_1250.fits'),
    ('16Cfh3ojWQTLUkFhcacOqC2BHrTa30FyE','planet3_00078_1250.fits'),
    ('1tnwGjjlIsBeD5x-aawbKfjZbWqY-d5pI','planet3_00654_1250.fits'),
    ('1MfPdRfQ9cmUPI4vi5-vjPsACU8eooWVP','planet3_00806_1250.fits'),
    ('19CluiSCmrwKud1KG8ja5msNzRHoEGwLd','planet3_00896_1250.fits'),
    ('1ueRxmJCQsxY6jH_YBT0FRoSI1ultUjW3','planet4_00442_1250.fits'),
    ('16vDbws00CEney0sNydmQz3DO0XUJyOdS','planet5_00354_1250.fits'),
    ('1qSURbZehtw14V_AxFA869Qwnd3Jn1lX8','planet5_00482_1250.fits'),
    ('18RQ63cgioqi15FADcNALDfOtWe_32p4h','planet5_00882_1250.fits'),
    ('1tdvJPPTzguNx2HN0CrJbYAC7b_-cFSqa','planet6_00370_1250.fits'),
    ('1JEAT8hwuBIjxHNoYXc508gXZmHRQkoQc','planet6_00566_1250.fits'),
    ('15Zwbk_oTMwacbKVrUEXrk30SXpOqTQg4','planet6_00638_1250.fits'),
    ('1_OX_qXaLBphjrWfWBavPDfXcbajL0IyR','planet6_00704_1250.fits'),
    ('15vw6snj_3vJAKoa-Wypfh5PpUm3CDhXt','planet7_00202_1250.fits'),
    ('1VvAFMc0rPyRLthIIB_g2fceeER2C2Vrl','planet7_00730_1250.fits'),
    ('1NIdz6hagTaac1fiaaL6X1mVN6IECebRO','planet8_00100_1250.fits'),
    ('1dnLUogfuLbcaOPSxaMMZirsuqludLr5s','planet8_00128_1250.fits'),
    ('1AHps7TLYf6YigEpAavJvZvm_RIdDx9l4','planet8_00214_1250.fits'),
    ('1Ppjove6P5oZaa66Fc8qChn0q19yeNKB8','planet8_00248_1250.fits'),
    ('15kGgdA2lwPb94OkxySnmZuAylZNNlkW_','planet8_00320_1250.fits'),
    ('147Z7EIL-N3oNRdBDexDM9G74Pw0N1kKz','planet9_00440_1250.fits'),
    ('13u5wrOUmNsV2pbIgeLUceh6mHo59AMuR','planet9_00916_1250.fits'),
    ('1gP74BZq8x5DLMMR9zeJorSMemP2XedIb','planet10_00710_1250.fits'),
    ('1VGB-QU5LXYl6W5K-FuohH2ifZYcrGcQE','planet10_00816_1250.fits'),
    ('1DVTvdurfgZftq_3_WdQ_umlO8HXt5kcy','planet11_00010_1250.fits'),
    ('1In1uYkdsuv2-a-e9zDc6sszIcxD1SBX4','planet11_00700_1250.fits'),
    ('1keUUPRhMIDz6RU95m4qVlGLbFUVxZDAo','planet11_00886_1250.fits'),
    ('13dTMy4tiVEyJlmVoPolEXmNw2-B3ttCb','planet13_00098_1250.fits'),
    ('16iEuhhlImAjAz04_uZmRpUqivAOf8C8L','planet13_00302_1250.fits'),
    ('1PN_0k8K_bK9KxKfNX1S5hqAaQl6EHvFt','planet13_00324_1250.fits'),
    ('1Lr29hb6C_0aPUzGLrJljt5Oee4Mcvft0','planet14_00314_1250.fits'),
    ('1w-bsj4rYUQvxZoifGmKroyKEcVRvg0Hz','planet14_00528_1250.fits'),
    ('1PLmh2bP3REZC-uu6LZ80HNq8EEUZeUtJ','planet14_00804_1250.fits'),
    ('1PvOVtu39pYxm_bF0pBoRnrsc47uTM5kT','planet14_00914_1250.fits'),
    ('1k-ySLE_01CpT3UdJyN6RM9T6sUcFZdZ8','planet15_00186_1250.fits'),
    ('1cELroa1QW-BR8GWzUdC4o-GHasPKhooe','planet16_00614_1250.fits'),
    ('1uRq-99mrwMH1K6mt_iD4LjQc-PNH-LSt','planet17_00162_1250.fits'),
    ('1GnZ8KpVAzxHPcJNmoNOAB5MtI4TBVnbj','planet17_00506_1250.fits'),
    ('1Sy4ti2sE-Za1dx_GL4H6DYN5lxVV_wPC','planet17_00648_1250.fits'),
    ('1MHtp3TRZwngGpH8mzow_QmtqY4ZwkD03','planet18_00114_1250.fits'),
    ('1qPKCXvV2Ll_2sWTq6hgT8enOsBRUqJRn','planet18_00942_1250.fits'),
    ('1AjRb_tfw7O5U9SdsXZAhOYOgKl1fYWwn','planet18_00992_1250.fits'),
    ('1VTsTjIkL2OCnsDs_CtfL3NyNh2R_D-yi','planet19_00914_1250.fits'),
    ('1RDzcTNStk22fFL6Pbqb67uG67f0tDFuG','planet20_00230_1250.fits'),
    ('1GJAn6PD8q1IiBICu5jEXMfypAR1Akqww','planet20_00336_1250.fits'),
]

existing = set(os.path.basename(f) for f in glob.glob(f'{DATA_DIR}/*.fits'))
to_download = [(fid,fn) for fid,fn in FILE_IDS if fn not in existing]

if to_download:
    print(f'Downloading {len(to_download)} missing files...')
    for fid, fname in to_download:
        gdown.download(id=fid, output=os.path.join(DATA_DIR, fname), quiet=False)

fits_files = sorted(glob.glob(f'{DATA_DIR}/*.fits'))
print(f'Found {len(fits_files)} FITS files')

In [ ]:
def load_fits(path):
    """Load FITS image with proper uint8 handling"""
    with fits.open(path) as h:
        # Convert to float32 IMMEDIATELY - fixes |u1 error
        d = h[0].data.astype(np.float32)

    # Handle different cube dimensions
    if d.ndim == 5:  # (1,1,1,600,600)
        img = d[0,0,0]
    elif d.ndim == 4:  # (4,1,1,600,600)
        img = d[0,0]
    elif d.ndim == 3:
        img = d[0]
    else:
        img = d

    return img.astype(np.float32)

# Inspect one file to confirm shape and value range
sample_raw = load_fits(fits_files[0])  # ✅ Fixed: fits_files (plural)
print(f'Raw shape : {sample_raw.shape}')
print(f'Raw range : min={sample_raw.min():.4e}  max={sample_raw.max():.4e}')
print(f'Non-zero  : {np.count_nonzero(sample_raw)} / {sample_raw.size}')
print(f'NaN count : {np.isnan(sample_raw).sum()}')

def preprocess(img, size=128):
    from PIL import Image
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    # Clip extreme outlier pixels (e.g. beam artefacts)
    img = np.clip(img, np.percentile(img, 0.5), np.percentile(img, 99.5))
    # Min-max normalize to [0, 1]
    lo, hi = img.min(), img.max()
    if hi > lo:
        img = (img - lo) / (hi - lo)
    # Resize to target_size
    pil = Image.fromarray((img * 255).astype(np.uint8))
    return np.array(pil.resize((size, size), Image.LANCZOS), dtype=np.float32) / 255.0


images, valid_files = [], []
for f in tqdm(fits_files):  
    try:
        img = preprocess(load_fits(f))
        images.append(img)
        valid_files.append(f)
    except Exception as e:
        print(f'Skip {os.path.basename(f)}: {str(e)}')
        continue

images = np.array(images)
print(f'Loaded {len(images)} images, shape: {images.shape}')

# SAFE value range print
if len(images) > 0:
    print(f'Value range: {images.min():.4f} – {images.max():.4f}')
else:
    print("No images loaded!")

# Plot samples
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(images): break
    ax.imshow(images[i], cmap='inferno', origin='lower', vmin=0, vmax=1)
    ax.axis('off')
    name = os.path.basename(valid_files[i]).replace('_1250.fits','')
    ax.set_title(name, fontsize=7)
plt.suptitle('Sample Disks', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
class DiskDataset(Dataset):
    def __init__(self,imgs): self.t=torch.FloatTensor(imgs).unsqueeze(1)
    def __len__(self): return len(self.t)
    def __getitem__(self,i): return self.t[i]

class Autoencoder(nn.Module):
    def __init__(self, latent=32):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1,32,3,2,1),  nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32,64,3,2,1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64,128,3,2,1),nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128,256,3,2,1),nn.BatchNorm2d(256),nn.ReLU(),
            nn.Flatten(), nn.Linear(256*8*8,512), nn.ReLU(), nn.Linear(512,latent)
        )
        self.dec = nn.Sequential(
            nn.Linear(latent,512), nn.ReLU(), nn.Linear(512,256*8*8), nn.ReLU(),
            nn.Unflatten(1,(256,8,8)),
            nn.ConvTranspose2d(256,128,4,2,1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128,64,4,2,1),  nn.BatchNorm2d(64),  nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1),   nn.BatchNorm2d(32),  nn.ReLU(),
            nn.ConvTranspose2d(32,1,4,2,1),    nn.Sigmoid()
        )
    def encode(self,x): return self.enc(x)
    def forward(self,x): z=self.enc(x); return self.dec(z),z

model = Autoencoder(latent=32).to(DEVICE)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
loader = DataLoader(DiskDataset(images), batch_size=32, shuffle=True, num_workers=2)
opt   = optim.Adam(model.parameters(), lr=1e-3)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
crit  = nn.MSELoss()
losses = []

for ep in range(50):
    model.train(); el=0
    for x in loader:
        x=x.to(DEVICE); r,_=model(x); loss=crit(r,x)
        opt.zero_grad(); loss.backward(); opt.step(); el+=loss.item()
    sched.step(); losses.append(el/len(loader))
    if (ep+1)%10==0: print(f'Ep {ep+1}/50 loss={losses[-1]:.6f}')

torch.save(model.state_dict(),'disk_autoencoder.pth')
plt.figure(figsize=(8,4))
plt.plot(losses); plt.title('Training Loss'); plt.xlabel('Epoch'); plt.grid(alpha=.3); plt.show()

In [ ]:
model.load_state_dict(torch.load('disk_autoencoder.pth',map_location=DEVICE))
model.eval()
inf_loader = DataLoader(DiskDataset(images), batch_size=32, shuffle=False)
all_z = []
with torch.no_grad():
    for x in inf_loader:
        all_z.append(model.encode(x.to(DEVICE)).cpu().numpy())
latents = np.concatenate(all_z)
latents_s = StandardScaler().fit_transform(latents)
print('Latents:', latents.shape)

In [ ]:
K_range = range(2,10)
inertias, sils = [], []
for k in K_range:
    km = KMeans(k,random_state=SEED,n_init=10)
    lbl = km.fit_predict(latents_s)
    inertias.append(km.inertia_); sils.append(silhouette_score(latents_s,lbl))

best_k = list(K_range)[np.argmax(sils)]
fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].plot(K_range,inertias,'bo-'); axes[0].set_title('Elbow'); axes[0].set_xlabel('K'); axes[0].grid(alpha=.3)
axes[1].plot(K_range,sils,'ro-'); axes[1].axvline(best_k,color='g',ls='--',label=f'Best K={best_k}')
axes[1].set_title('Silhouette'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('cluster_selection.png',dpi=150,bbox_inches='tight'); plt.show()
print(f'Best K={best_k}, silhouette={max(sils):.4f}')

In [ ]:
kmeans = KMeans(best_k,random_state=SEED,n_init=20)
labels = kmeans.fit_predict(latents_s)
print(f'Silhouette: {silhouette_score(latents_s,labels):.4f}')
for c,n in zip(*np.unique(labels,return_counts=True)): print(f'  Cluster {c}: {n} disks')

In [ ]:
reducer = umap.UMAP(n_components=2,random_state=SEED,n_neighbors=15,min_dist=0.1)
emb = reducer.fit_transform(latents_s)
pca = PCA(2,random_state=SEED); pemb = pca.fit_transform(latents_s)

colors = plt.cm.Set1(np.linspace(0,1,best_k))
fig,axes=plt.subplots(1,2,figsize=(16,7))
for c in range(best_k):
    m=labels==c
    axes[0].scatter(emb[m,0],emb[m,1],c=[colors[c]],label=f'C{c}(n={m.sum()})',s=35,alpha=.8)
    axes[1].scatter(pemb[m,0],pemb[m,1],c=[colors[c]],label=f'C{c}',s=35,alpha=.8)
axes[0].set_title('UMAP Clusters',fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=.2)
axes[1].set_title(f'PCA ({sum(pca.explained_variance_ratio_):.1%} var)',fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=.2)
plt.tight_layout(); plt.savefig('umap_clusters.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
import json

input_path = r"C:\Users\VIREN PANDEY\Desktop\ML4SCI-GSoC-2026\EXXA\EXXA_GeneralTest_VirenPandey.ipynb"
output_path = r"C:\Users\VIREN PANDEY\Desktop\ML4SCI-GSoC-2026\EXXA\EXXA_GeneralTest_VirenPandey_clean.ipynb"

with open(input_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

nb.get("metadata", {}).pop("widgets", None)

for cell in nb.get("cells", []):
    if cell.get("cell_type") == "code":
        cell["outputs"] = []
        cell["execution_count"] = None

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=2)

print(f"Clean notebook saved to: {output_path}")

In [ ]:
fig,axes=plt.subplots(best_k,6,figsize=(18,3*best_k))
if best_k==1: axes=axes[np.newaxis,:]
for c in range(best_k):
    idx = np.where(labels==c)[0]
    dists = np.linalg.norm(latents_s[idx]-kmeans.cluster_centers_[c],axis=1)
    top6 = idx[np.argsort(dists)[:6]]
    for j,i in enumerate(top6):
        axes[c,j].imshow(images[i],cmap='inferno',origin='lower'); axes[c,j].axis('off')
        if j==0: axes[c,j].set_ylabel(f'Cluster {c}\n(n={len(idx)})',fontweight='bold',rotation=0,labelpad=60,va='center')
plt.suptitle('Representative Disks per Cluster',fontweight='bold')
plt.tight_layout(); plt.savefig('cluster_representatives.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
from scipy.signal import find_peaks

def disk_features(img):
    h,w=img.shape; cy,cx=h//2,w//2
    y,x=np.ogrid[:h,:w]; r=np.sqrt((x-cx)**2+(y-cy)**2).astype(int)
    max_r=min(cx,cy)
    radial=np.array([img[r==ri].mean() if (r==ri).any() else 0 for ri in range(max_r)])
    peaks,_=find_peaks(radial,height=radial.max()*.1,distance=5)
    flat=img.flatten(); cs=np.cumsum(np.sort(flat)[::-1])
    hlr=np.sqrt(np.searchsorted(cs,cs[-1]*.5)/np.pi)
    pr=np.argmax(radial); rm=(r>=max(0,pr-2))&(r<=pr+2)
    asym=img[rm].std()/(img[rm].mean()+1e-8)
    return [img.sum(), img.max(), hlr, len(peaks), asym]

feat_names = ['total_flux','peak','half_light_r','n_rings','asymmetry']
feats = np.array([disk_features(im) for im in tqdm(images)])

fig,axes=plt.subplots(1,5,figsize=(20,4))
for i,(nm,ax) in enumerate(zip(feat_names,axes)):
    for c in range(best_k):
        ax.hist(feats[labels==c,i],bins=20,alpha=.6,label='C'+str(c),color=colors[c])
    ax.set_title(nm.replace('_',' ')); ax.legend(fontsize=8); ax.grid(alpha=.2)
plt.suptitle('Feature Distributions per Cluster',fontweight='bold')
plt.tight_layout(); plt.savefig('feature_distributions.png',dpi=150,bbox_inches='tight'); plt.show()

header = 'Feature'.ljust(20) + ''.join('  Cluster ' + str(c) for c in range(best_k))
print(header)
for i,nm in enumerate(feat_names):
    row = nm.ljust(20) + ''.join(f'{feats[labels==c,i].mean():>10.3f}' for c in range(best_k))
    print(row)


In [ ]:
model.eval()
idx8=np.random.choice(len(images),8,replace=False)
with torch.no_grad():
    x=torch.FloatTensor(images[idx8]).unsqueeze(1).to(DEVICE)
    recons,_=model(x); recons=recons.cpu().numpy().squeeze(1)

fig,axes=plt.subplots(3,8,figsize=(20,8))
for j in range(8):
    o,r=images[idx8[j]],recons[j]
    for row,im,cmap in [(0,o,'inferno'),(1,r,'inferno'),(2,np.abs(o-r),'hot')]:
        axes[row,j].imshow(im,cmap=cmap,origin='lower'); axes[row,j].axis('off')
axes[0,0].set_ylabel('Original',fontweight='bold')
axes[1,0].set_ylabel('Reconstructed',fontweight='bold')
axes[2,0].set_ylabel('Residual',fontweight='bold')
plt.suptitle('Reconstruction Quality',fontweight='bold')
plt.tight_layout(); plt.savefig('reconstruction_quality.png',dpi=150,bbox_inches='tight'); plt.show()
print(f'Mean MSE: {np.mean((images[idx8]-recons)**2):.6f}')

In [ ]:
print('='*55)
print('SUMMARY — EXXA General Test')
print('='*55)
print(f'Disks loaded : {len(images)}')
print(f'Image size   : 128x128')
print(f'Latent dim   : 32')
print(f'Clusters (K) : {best_k}')
print(f'Silhouette   : {silhouette_score(latents_s,labels):.4f}')
for c,n in zip(*np.unique(labels,return_counts=True)): print(f'  Cluster {c}: {n} disks')